# 🐝 Heady Brain — GPU Runtime

**Chat directly with HeadyBrain. Keys load automatically — just run the cells.**

1. Run Cell 1 (loads keys)
2. Run Cell 2 (chat engine with **persistent conversation history**)
3. Run Cell 3 to talk!

### Commands
- `chat("msg")` — continues conversation
- `new_chat()` — fresh conversation
- `save_chat("name")` / `load_chat("name")` — persist across restarts
- `show_history()` — view conversation so far

In [ ]:
# ═══ Cell 1: Load Keys from Heady Memory ═══import base64 as _b, osdef _d(s): return _b.b64decode(s).decode()_vault = {    'gemini': [        (_d('QUl6YVN5RE9MNVlKX042Q0xlVk5ReXRCeURENm5aMEktS3I0WVJr'), 'HC-Heady'),        (_d('QUl6YVN5QllSWEtTckdDVXBfWUxhbGo5R0NRNnRtbjlNZENpVXNn'), 'HC-GCloud'),        (_d('QUl6YVN5Q3hkTlowQUpFVVoxUi1iQWljaDJIa28tRHpSb0w4RjVv'), 'HM-2'),        (_d('QUl6YVN5RENZTXJjYkVONTF1d0lteXR2RUtPeUVFbXVaZGc4TFBr'), 'HM-3'),        (_d('QUl6YVN5QkxUdTBoOVEwOUNyMDVfM19aal8zeWVudDVjTzNpYUhF'), 'HM-4'),        (_d('QUl6YVN5REIxU2MyUHpzUld4TldvaGNJQ0dIQXFRS001WXZSY1hv'), 'HM-Default'),    ],    'openai': [        (_d('c2stcHJvai1mb29fcDRGa2JCWklDTENRcFc4T3VmMlRLTlNZVlBrTnd5Q3lKUlBDQ29xVmFITWVLT3NKSF9Ha0h5bHhKR1JzZW85VHQySVo3QlQzQmxia0ZKT2ZXNThjTTh6LUFDMXY5dUt4YjdkelpoMjNLXy1XM0lmb0U5NkxCME44NXJlVEp2V1Mxa0llOHFxM2ZkTzlsdEVVUHJWOUNWVUE='), 'Seat1'),        (_d('c2stc3ZjYWNjdC0xNzB6NFhLYnI4dE0xTDBRbkpsMEMtS0lOeldaemFLM0E3RVpIaDJMRjNHczJqUUplZEpuallJRUFHb3A2OUwxbGY1WGwyajcyS1QzQmxia0ZKdTNHRnFZUVZxdDVuc2ZxdmlFNm1xUXVyUmlYM19DbXk1TVYxRjlFdUVzSU1odHNZaTVyNmZ2YURnWGRYWWN4VGZ0cUZWZXctd0E='), 'SvcAcct'),    ],    'anthropic': [        (_d('c2stYW50LWFwaTAzLXN4cnZjRkVmdzBZMkJVWnVySVNidmhaS0N0OWg3bTNyRDZXa3hUaDJKekR4U2N0MlFENjZTZTEybjRjbkIxUVM1Nk1IYWNQWFgwVERXUG8wM0ppSU1RLWRRYkJid0FB'), 'Claude-API03'),    ],    'huggingface': [        (_d('aGZfdW1IdXlLV1pVU2tXaG54U3RaQVFORUdtSFNXenBlU0dLaw=='), 'HF-Seat1'),        (_d('aGZfT3lwU0JwSVlNaERrR3BEVkZoYVZJSUxPRmVnUlpWV3hPdA=='), 'HF-Seat2'),    ],    'perplexity': [],}def _get(name):    try:        from google.colab import userdata        v = userdata.get(name)        if v: return v    except: pass    return os.environ.get(name)for name, provider in [('GEMINI_API_KEY','gemini'),('OPENAI_API_KEY','openai'),('ANTHROPIC_API_KEY','anthropic'),('HF_TOKEN','huggingface'),('PERPLEXITY_API_KEY','perplexity')]:    v = _get(name)    if v and not any(k==v for k,_ in _vault.get(provider,[])): _vault[provider].append((v, f'env:{name}'))KEYS = _vaulttotal = sum(len(v) for v in KEYS.values())print(f'🐝 Heady Memory: {total} keys loaded automatically')for p, ks in KEYS.items():    if ks: print(f'  ✅ {p}: {len(ks)} key(s)')    else:  print(f'  ⬜ {p}: none')

In [ ]:
# ═══ Cell 2: Chat Engine with Persistent Conversation History ═══import json, urllib.request, time, concurrent.futures, os, uuidSYSTEM = """You are HeadyBrain, the sovereign AI reasoning engine of the Heady ecosystem.3 accounts (HeadySystems, HeadyConnection, HeadyMe), 5 domains, HuggingFace Spaces, Cloud Run.Be helpful, precise, and actionable."""conversation_history = []session_id = str(uuid.uuid4())[:8]MAX_HISTORY = 50SESSIONS_DIR = '/content/heady_sessions'def _trim_history():    global conversation_history    if len(conversation_history) > MAX_HISTORY:        conversation_history = conversation_history[-MAX_HISTORY:]def new_chat():    global conversation_history, session_id    conversation_history = []    session_id = str(uuid.uuid4())[:8]    print(f'🆕 New conversation started (session: {session_id})')def show_history():    if not conversation_history:        print('�� No conversation history yet.')        return    print(f'📜 Conversation ({len(conversation_history)} messages):')    for i, msg in enumerate(conversation_history):        role = '👤 You' if msg['role'] == 'user' else '🐝 Heady'        preview = msg['content'][:200]        print(f'  {i+1}. {role}: {preview}')def save_chat(name='auto'):    os.makedirs(SESSIONS_DIR, exist_ok=True)    path = os.path.join(SESSIONS_DIR, f'{name}.json')    data = {'session_id': session_id, 'history': conversation_history, 'saved_at': time.strftime('%Y-%m-%dT%H:%M:%S')}    with open(path, 'w') as f:        json.dump(data, f, indent=2)    print(f'💾 Saved {len(conversation_history)} messages to {path}')def load_chat(name='auto'):    global conversation_history, session_id    path = os.path.join(SESSIONS_DIR, f'{name}.json')    if not os.path.exists(path):        print(f'❌ No saved chat found at {path}')        return    with open(path) as f:        data = json.load(f)    conversation_history = data.get('history', [])    session_id = data.get('session_id', name)    print(f'📂 Loaded {len(conversation_history)} messages (session: {session_id})')def _call(provider, key, message):    if provider == 'gemini':        contents = []        for msg in conversation_history:            role = 'model' if msg['role'] == 'assistant' else 'user'            contents.append({'role': role, 'parts': [{'text': msg['content']}]})        contents.append({'role': 'user', 'parts': [{'text': message}]})        url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={key}'        data = json.dumps({            'systemInstruction': {'parts': [{'text': SYSTEM}]},            'contents': contents,            'generationConfig': {'maxOutputTokens': 8192}        }).encode()        req = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})        d = json.loads(urllib.request.urlopen(req, timeout=120).read())        return d['candidates'][0]['content']['parts'][0]['text']    elif provider == 'openai':        messages = [{'role': 'system', 'content': SYSTEM}]        messages.extend(conversation_history)        messages.append({'role': 'user', 'content': message})        data = json.dumps({'model': 'gpt-4o', 'messages': messages, 'max_tokens': 8192}).encode()        req = urllib.request.Request('https://api.openai.com/v1/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})        d = json.loads(urllib.request.urlopen(req, timeout=120).read())        return d['choices'][0]['message']['content']    elif provider == 'anthropic':        messages = list(conversation_history)        messages.append({'role': 'user', 'content': message})        data = json.dumps({'model': 'claude-sonnet-4-20250514', 'max_tokens': 8192, 'system': SYSTEM, 'messages': messages}).encode()        req = urllib.request.Request('https://api.anthropic.com/v1/messages', data=data, headers={'Content-Type': 'application/json', 'x-api-key': key, 'anthropic-version': '2023-06-01'})        d = json.loads(urllib.request.urlopen(req, timeout=120).read())        return d['content'][0]['text']    elif provider == 'perplexity':        messages = [{'role': 'system', 'content': SYSTEM}]        messages.extend(conversation_history)        messages.append({'role': 'user', 'content': message})        data = json.dumps({'model': 'sonar-pro', 'messages': messages, 'max_tokens': 4096}).encode()        req = urllib.request.Request('https://api.perplexity.ai/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})        d = json.loads(urllib.request.urlopen(req, timeout=120).read())        return d['choices'][0]['message']['content']    elif provider == 'huggingface':        messages = list(conversation_history)        messages.append({'role': 'user', 'content': message})        data = json.dumps({'model': 'deepseek/deepseek-r1-0528', 'messages': messages, 'max_tokens': 4096, 'stream': False}).encode()        req = urllib.request.Request('https://router.huggingface.co/novita/v3/openai/chat/completions', data=data, headers={'Content-Type': 'application/json', 'Authorization': f'Bearer {key}'})        d = json.loads(urllib.request.urlopen(req, timeout=120).read())        return d['choices'][0]['message']['content']def chat(message, provider='auto'):    if provider == 'all':        return deep_research(message)    if provider == 'auto':        for p in ['gemini', 'openai', 'anthropic', 'huggingface', 'perplexity']:            if KEYS.get(p):                provider = p                break        if provider == 'auto':            print('❌ No keys! Run Cell 1.')            return    keys = KEYS.get(provider, [])    if not keys:        print(f'❌ No keys for {provider}')        return    for key, label in keys:        try:            s = time.time()            r = _call(provider, key, message)            elapsed = int((time.time() - s) * 1000)            conversation_history.append({'role': 'user', 'content': message})            conversation_history.append({'role': 'assistant', 'content': r})            _trim_history()            ctx = f'{len(conversation_history)//2} turns'            print(f'🐝 HeadyBrain [{provider}/{label}] ({elapsed}ms, {ctx}):{r}')            return r        except Exception as e:            print(f'  ⚠ {label}: {str(e)[:80]}, trying next...')    print(f'❌ All keys failed for {provider}')def deep_research(message):    print('🔬 Deep Research: querying ALL providers...')    tasks = [(p, ks[0][0], ks[0][1]) for p, ks in KEYS.items() if ks]    results = []    start = time.time()    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as pool:        futs = {pool.submit(_call, p, k, message): (p, l) for p, k, l in tasks}        for f in concurrent.futures.as_completed(futs):            p, l = futs[f]            try:                r = f.result(timeout=120)                print(f'✅ {p}/{l}: {len(r)} chars')                results.append((p, l, r))            except Exception as e:                print(f'❌ {p}/{l}: {str(e)[:60]}')    conversation_history.append({'role': 'user', 'content': message})    if results:        best = max(results, key=lambda x: len(x[2]))        conversation_history.append({'role': 'assistant', 'content': best[2]})    _trim_history()    total_ms = int((time.time()-start)*1000)    print(f'═══ {len(results)}/{len(tasks)} ({total_ms}ms) ═══')    for p, l, r in results:        print(f'── {p.upper()}/{l} ({len(r)} chars) ──{r[:3000]}')        if len(r) > 3000:            print(f'... [{len(r)-3000} more]')        print()    return resultsprint(f'💬 Chat ready! Session: {session_id}')print('  chat("your message")              → auto (continues conversation)')print('  chat("msg", "openai")              → GPT-4o')print('  chat("msg", "anthropic")           → Claude')print('  chat("msg", "all")                 → ALL providers')print('  new_chat()                         → fresh conversation')print('  show_history()                     → view history')print('  save_chat("name") / load_chat("name")')

In [ ]:
# ═══ Cell 3: 💬 CHAT ═══
chat("What should we prioritize to get Heady to enterprise grade?")

In [ ]:
# ═══ Cell 4: 🔬 DEEP RESEARCH ═══
deep_research("Enterprise best practices for multi-provider AI with autonomous operations")

In [ ]:
# ═══ Cell 5: 🔄 Interactive Chat (with persistent history) ═══print('🐝 HeadyBrain — type and talk! History is maintained.')print('Prefix: @openai @claude @gemini @hf @all')print('Commands: /new /save [name] /load [name] /history /quit')print()while True:    try:        msg = input('You: ')        if msg.lower() in ('quit', 'exit', 'q', '/quit'):            break        if not msg.strip():            continue        if msg.strip() == '/new':            new_chat()            continue        if msg.strip() == '/history':            show_history()            continue        if msg.strip().startswith('/save'):            parts = msg.strip().split(' ', 1)            name = parts[1] if len(parts) > 1 else 'auto'            save_chat(name)            continue        if msg.strip().startswith('/load'):            parts = msg.strip().split(' ', 1)            name = parts[1] if len(parts) > 1 else 'auto'            load_chat(name)            continue        p = 'auto'        for pfx, prov in [('@openai ', 'openai'), ('@claude ', 'anthropic'), ('@gemini ', 'gemini'), ('@hf ', 'huggingface'), ('@all ', 'all')]:            if msg.startswith(pfx):                p, msg = prov, msg[len(pfx):]                break        print()        chat(msg, p)        print()    except KeyboardInterrupt:        print('🛑 Done.')        break